In [10]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated, Literal
from langchain_groq import ChatGroq
from pydantic import BaseModel,Field
from dotenv import load_dotenv
import operator
from langchain_core.messages import SystemMessage, HumanMessage, ChatMessage
from pydantic import BaseModel, Field

In [5]:
load_dotenv()

True

In [6]:
generate_llm = model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)
evaluator_llm = model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)
optimizer_llm = model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [23]:
class TweetState(TypedDict):
    topic: str
    tweet: str
    evaluation: Literal["approved","needs_improvement","rejected"]
    feedback: str
    iteration: int
    max_iterations: int




In [9]:
def generate_tweet(state: TweetState) -> TweetState:
     messages = [
        SystemMessage(content="You are a funny and clever Twitter/X influencer."),
        HumanMessage(content=f"""
Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

Rules:
- Do NOT use question-answer format.
- Max 280 characters.
- Use observational humor, irony, sarcasm, or cultural references.
- Think in meme logic, punchlines, or relatable takes.
- Use simple, day to day english
""")
    ]

     response = generate_llm.invoke(messages).content

     return {'tweet': response}

In [11]:
class EvaluationResult(BaseModel):
    evaluation: Literal["approved","needs_improvement"] = Field(..., description="The evaluation result for the generated tweet.")
    feedback: str = Field(..., description="Constructive feedback for improving the tweet if it is not approved.")

In [12]:
structured_llm = evaluator_llm.with_structured_output(EvaluationResult)

In [13]:
def evaluate_tweet(state: TweetState) -> TweetState:
    messages = [
    SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."),
    HumanMessage(content=f"""
Evaluate the following tweet:

Tweet: "{state['tweet']}"

Use the criteria below to evaluate the tweet:

1. Originality – Is this fresh, or have you seen it a hundred times before?  
2. Humor – Did it genuinely make you smile, laugh, or chuckle?  
3. Punchiness – Is it short, sharp, and scroll-stopping?  
4. Virality Potential – Would people retweet or share it?  
5. Format – Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

Auto-reject if:
- It's written in question-answer format (e.g., "Why did..." or "What happens when...")
- It exceeds 280 characters
- It reads like a traditional setup-punchline joke
- Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)

### Respond ONLY in structured format:
- evaluation: "approved" or "needs_improvement"  
- feedback: One paragraph explaining the strengths and weaknesses 
""")
]
    response = structured_llm.invoke(messages)

    return {'evaluation': response.evaluation, 'feedback': response.feedback}

In [14]:
def optimize_tweet(state: TweetState) -> TweetState:
    messages = [
        SystemMessage(content="You punch up tweets for virality and humor based on given feedback."),
        HumanMessage(content=f"""
Improve the tweet based on this feedback:
"{state['feedback']}"

Topic: "{state['topic']}"
Original Tweet:
{state['tweet']}

Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
""")
    ]
    response = optimizer_llm.invoke(messages).content
    iteration = state['iteration'] + 1
    return {'tweet': response, 'iteration': iteration}

In [28]:
def judge(state: TweetState):
    if state['evaluation'] == 'approved':
        return 'approved'
    elif state['iteration'] >= state['max_iterations']:
        return 'approved'
    else:
        return 'needs_improvement'

In [31]:
graph = StateGraph(TweetState)
graph.add_node('generate',generate_tweet)
graph.add_node('evaluate',evaluate_tweet)
graph.add_node('optimize',optimize_tweet)
graph.add_edge(START, 'generate')
graph.add_edge('generate', 'evaluate')
graph.add_conditional_edges('evaluate', judge, {'approved': END, 'needs_improvement': 'optimize'})
graph.add_edge('optimize', 'evaluate')
workflow = graph.compile()

In [36]:
initial_state = {
    'topic': "IIT",
    'iteration': 1,
    'max_iterations': 3
}
workflow.invoke(initial_state)

{'topic': 'IIT',
 'tweet': '"IIT: where you pay to get rejected by girls, and also by placements" #IITlife',
 'evaluation': 'approved',
 'feedback': 'This tweet is a great example of a well-crafted joke that effectively uses humor to comment on the IIT experience. The originality of the tweet lies in its ability to poke fun at the dual rejections that IIT students face, making it relatable and fresh. The humor is spot on, as it cleverly highlights the struggles of IIT life in a lighthearted way. The tweet is also punchy, short, and scroll-stopping, making it perfect for the Twitter format. Additionally, the use of the hashtag #IITlife increases its virality potential, as it can resonate with a specific audience and encourage retweets and shares. Overall, the tweet is well-formed, concise, and effectively uses humor to make a point, making it a great example of a tweet that is both funny and engaging.',
 'iteration': 1,
 'max_iterations': 3}